Name : N. Gifty Taniksha  
Roll No.: 2503A51107

# Lab Assignment 6 : Interactive Visualization with Plotly and Multivariate Data Analysis

**Dataset:** `Ecommerce_Customer_Journey_India_2024 (1).xlsx` (`Raw_Dataset` sheet) — 1,500 e-commerce order records from India, Jan–Dec 2024, covering 28 states, 21 products across 7 categories, and 4 payment methods.


| Task | Chart | Type of analysis |
|---|---|---|
| 1.1 | Histogram of Revenue with range slider | Univariate |
| 1.2 | Scatter: Unit Price vs. Delivery Days, coloured by Return Status | Multivariate |
| 1.3 | Box plot: Revenue by Payment Method, all points shown | Bivariate |
| 1.4 | 3D scatter: Quantity, Unit Price, Revenue | Multivariate |
| 2.1 | Animated scatter: Unit Price vs. Revenue, one frame per month | Temporal + multivariate |
| 2.2 | Scatter with dropdown to switch the y-axis metric | Multivariate |


## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"
px.defaults.height = 560
px.defaults.width = 980

print("Plotly version:", __import__("plotly").__version__)

Plotly version: 7.1.0


## 1. Load and inspect the data

In [ ]:
df = pd.read_excel("Ecommerce_Customer_Journey_India_2024 (1).xlsx", sheet_name="Raw_Dataset")
print("Shape:", df.shape)
df.head()

Shape: (1500, 16)


,Order_ID,Order_Date,Customer_ID,State,Season,Product_Category,Product_Name,Quantity,Unit_Price,Discount,Payment_Method,Delivery_Days,Customer_Rating,Revenue,Customer_Type,Return_Status
0,501135,2024-08-12,20229,West Bengal,Monsoon,Footwear,Sports Shoes,5,1913.72,0.0,Wallet,6.0,4.2,9568.60,New,Returned
1,501448,2024-09-21,20572,Karnataka,Monsoon,Accessories,Umbrella,3,1494.41,NaN,Wallet,4.0,3.6,4483.23,Returning,Not Returned
2,500988,2024-07-01,20564,Andhra Pradesh,Monsoon,Accessories,Umbrella,1,1168.97,0.0,UPI,7.0,3.3,1168.97,Returning,Returned
3,501212,2024-05-09,20476,Uttarakhand,Summer,Fashion,T-Shirt,1,1279.36,0.1,Card,5.0,4.0,1151.42,New,Not Returned
4,500536,2024-10-11,20115,Maharashtra,Festive,Electronics,Smartphone,1,12141.89,0.2,Wallet,2.0,4.8,9713.51,Returning,Not Returned


In [ ]:
# Data types and missing values per column
summary = pd.DataFrame({"dtype": df.dtypes.astype(str),
                        "missing": df.isna().sum(),
                        "missing_%": (df.isna().mean() * 100).round(1)})
summary

,dtype,missing,missing_%
Order_ID,int64,0,0.0
Order_Date,datetime64[us],0,0.0
Customer_ID,int64,0,0.0
State,str,0,0.0
Season,str,8,0.5
Product_Category,str,0,0.0
Product_Name,str,0,0.0
Quantity,int64,0,0.0
Unit_Price,float64,0,0.0
Discount,float64,10,0.7


In [ ]:
df[["Quantity", "Unit_Price", "Discount", "Delivery_Days", "Customer_Rating", "Revenue"]].describe().round(2)

,Quantity,Unit_Price,Discount,Delivery_Days,Customer_Rating,Revenue
count,1500.00,1500.00,1490.00,1488.00,1482.00,1500.00
mean,2.94,7928.16,0.12,5.05,3.99,13306.12
std,1.57,13960.41,0.08,2.10,0.63,21634.41
min,1.00,120.04,0.00,1.00,1.90,169.04
25%,1.00,937.02,0.05,4.00,3.50,2160.66
50%,3.00,2149.84,0.10,5.00,4.00,5611.52
75%,5.00,4671.45,0.20,6.00,4.50,14201.27
max,5.00,82560.16,0.30,10.00,5.00,196252.61


## 2. Cleaning (as per the dataset guide)

* **Duplicates:** the sheet carries 20 exact duplicate copies on top of 1,480 unique orders — dropped first.
* **Season:** 8 rows have a missing `Season`; it is fully determined by the month of `Order_Date` (Winter: Dec–Feb, Summer: Mar–Jun, Monsoon: Jul–Sep, Festive: Oct–Nov), so it is recalculated deterministically.
* **Discount:** 10 rows are missing; since `Revenue = Quantity × Unit_Price × (1 − Discount)`, it can be recovered exactly as `1 − Revenue / (Quantity × Unit_Price)`, rounded and clipped to the valid 0–0.30 range.
* **Delivery_Days:** 12 rows are missing; filled with the **median** delivery days for the same `State` (robust to long-tail delays), rounded to a whole day.
* **Customer_Rating:** 18 rows are missing; left as `NaN` — a blank does not mean zero stars, so it is excluded from averages rather than imputed.
* **Payment_Method:** 12 rows are missing; labelled `'Unrecorded'` so these orders stay in the denominator instead of being dropped or guessed.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

# Fixed category order and colours, reused in every chart for consistency
SEASON_MONTHS = {12: "Winter", 1: "Winter", 2: "Winter",
                 3: "Summer", 4: "Summer", 5: "Summer", 6: "Summer",
                 7: "Monsoon", 8: "Monsoon", 9: "Monsoon",
                 10: "Festive", 11: "Festive"}
df["Season"] = df["Order_Date"].dt.month.map(SEASON_MONTHS)

SEASON_ORDER = ["Winter", "Summer", "Monsoon", "Festive"]
SEASON_COLORS = {"Winter": "#1f77b4", "Summer": "#ff7f0e", "Monsoon": "#2ca02c", "Festive": "#9467bd"}

# Recover missing discount from Revenue = Quantity * Unit_Price * (1 - Discount)
mask = df["Discount"].isna()
recovered = (1 - df.loc[mask, "Revenue"] / (df.loc[mask, "Quantity"] * df.loc[mask, "Unit_Price"])).round(2)
df.loc[mask, "Discount"] = recovered.clip(0, 0.30)

# Fill missing delivery days with the median for the same state
df["Delivery_Days"] = df.groupby("State")["Delivery_Days"].transform(lambda s: s.fillna(round(s.median())))

# Payment method: label missing as Unrecorded (keep rows, do not guess)
df["Payment_Method"] = df["Payment_Method"].fillna("Unrecorded")

# Customer_Rating: left as NaN on purpose (see note above)
df["Order_Date"] = pd.to_datetime(df["Order_Date"])
df["Month"] = df["Order_Date"].dt.month_name()
MONTH_ORDER = ["January", "February", "March", "April", "May", "June",
              "July", "August", "September", "October", "November", "December"]

df = df.sort_values("Order_Date").reset_index(drop=True)
print("Remaining missing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nSeason counts:\n", df["Season"].value_counts().reindex(SEASON_ORDER))

Shape after removing duplicates: (1480, 16)
Remaining missing values:
 Customer_Rating    18
dtype: int64

Season counts:
 Season
Winter     330
Summer     467
Monsoon    330
Festive    353
Name: count, dtype: int64


In [ ]:
# Quick check: rating availability by month (ratings are genuinely optional feedback, not random noise)
(df.assign(rated=df["Customer_Rating"].notna())
   .groupby("Month")["rated"].mean().mul(100).round(0)
   .reindex(MONTH_ORDER).rename("% orders with a rating").to_frame().T)

Month,January,February,March,April,May,June,July,August,September,October,November,December
% orders with a rating,99.0,99.0,100.0,98.0,99.0,98.0,98.0,98.0,100.0,99.0,97.0,100.0


Rating coverage is fairly stable across months (roughly 97–99%), so the ~1.3% of missing ratings look like ordinary unsubmitted feedback rather than a seasonal collection gap — safe to exclude from rating averages without biasing any one month.

---
# Task 1: Core Interactive Charts (Session 11)

### 1.1 Univariate histogram of Revenue with a range slider

In [ ]:
fig1 = px.histogram(df, x="Revenue",
                    title="Distribution of order revenue (India, 2024)",
                    labels={"Revenue": "Revenue (INR)"})
fig1.update_traces(xbins=dict(start=0, end=df["Revenue"].max() + 500, size=500))
fig1.update_layout(yaxis_title="Number of orders", bargap=0.05,
                   height=560, width=980, margin=dict(t=70, b=60),
                   xaxis=dict(rangeslider=dict(visible=True, thickness=0.12)))
# Note: with a range slider, Plotly always draws the x-axis title BELOW the slider.
fig1.show()

*Drag the handles of the range slider to zoom into, for example, orders above ₹20,000. Revenue is right-skewed: most orders cluster at lower values (driven by `Quantity` 1–5 and the product price ranges), with a long tail of high-value orders from expensive categories like Electronics and Home Appliances.*

### 1.2 Multivariate scatter: Unit Price vs. Delivery Days, coloured by Return Status

In [ ]:
RETURN_COLORS = {"Returned": "#d62728", "Not Returned": "#2ca02c"}
fig2 = px.scatter(df, x="Unit_Price", y="Delivery_Days", color="Return_Status",
                  color_discrete_map=RETURN_COLORS,
                  hover_name="Product_Name",
                  hover_data={"State": True, "Payment_Method": True,
                              "Order_Date": "|%d %b %Y", "Revenue": ":.2f",
                              "Return_Status": False},
                  opacity=0.6,
                  labels={"Unit_Price": "Unit price (INR)", "Delivery_Days": "Delivery days",
                          "Return_Status": "Return status"},
                  title="Unit price vs. delivery days, coloured by return status")
fig2.update_layout(height=560, width=980, margin=dict(t=70))
fig2.update_traces(marker=dict(size=7))
fig2.show()

*There is no strong visual separation between returned and not-returned orders along price or delivery time — returns are spread fairly evenly across both axes, suggesting return behaviour here is driven more by product fit/quality than by price or delivery delay alone.*

### 1.3 Box plot of Revenue by Payment Method, with individual points

In [ ]:
PAY_ORDER = sorted(df["Payment_Method"].unique(), key=lambda x: x == "Unrecorded")
fig3 = px.box(df, x="Payment_Method", y="Revenue", color="Payment_Method",
              category_orders={"Payment_Method": PAY_ORDER},
              points="all", hover_name="Product_Name",
              hover_data={"Order_Date": "|%d %b %Y", "State": True, "Payment_Method": False},
              labels={"Payment_Method": "Payment method", "Revenue": "Revenue (INR)"},
              title="Revenue by payment method (all orders shown)")
fig3.update_traces(marker=dict(size=5, opacity=0.5), jitter=0.4)
fig3.update_layout(showlegend=False, height=560, width=980, margin=dict(t=70))
fig3.show()

In [ ]:
# Numbers behind the box plot
df.groupby("Payment_Method")[["Revenue", "Quantity"]].median().reindex(PAY_ORDER).round(2)

,Revenue,Quantity
Payment_Method,,
Card,5713.79,3.0
Cash on Delivery,5313.33,3.0
Wallet,6672.82,3.0
UPI,5285.36,3.0
Unrecorded,4893.89,3.0


*Median revenue is broadly similar across UPI, Card, Wallet and Cash on Delivery — payment method looks like a customer-preference choice, not a proxy for order size. The small `Unrecorded` group (missing payment data) sits in the same range, so excluding it would not bias any of the other medians.*

### 1.4 3D scatter of Quantity, Unit Price and Revenue

In [ ]:
fig4 = px.scatter_3d(df, x="Unit_Price", y="Quantity", z="Revenue", color="Product_Category",
                     hover_name="Product_Name", opacity=0.75,
                     labels={"Unit_Price": "Unit price (INR)", "Quantity": "Quantity",
                             "Revenue": "Revenue (INR)", "Product_Category": "Category"},
                     title="Unit price, quantity and revenue in 3D (rotate and zoom)")
fig4.update_traces(marker=dict(size=4))
fig4.update_layout(height=780, width=980, margin=dict(t=70))
fig4.show()

*Revenue rises diagonally with both price and quantity, as expected from the formula `Revenue = Quantity x Unit_Price x (1 - Discount)`. Rotating the view separates categories cleanly along the price axis: Electronics and Home Appliances sit at high unit price, while Accessories and Beauty cluster near the low-price, low-revenue corner.*

---
# Task 2: Animation and Dropdown Controls (Session 12)

### 2.1 Animated scatter: Unit price vs. Revenue, one frame per month

The dataset spans 12 months with roughly 120 orders each, so one frame per month is a natural granularity (12 frames, each with a reasonable sample size).

**Plotly animation detail:** `Season` is fully determined by the month of `Order_Date` (see the cleaning step above), so by construction every single month belongs to exactly one season. If colour is mapped straight to `Season`, each frame's legend would show only that one colour and the other three would disappear while playing. The fix below adds an invisible placeholder row (`NaN` coordinates) for every `(Month, Season)` pair that does not naturally occur — the same fixed-trace trick used for the earthquake `alert` animation — so all four season colours stay listed in the legend throughout the whole animation.

In [ ]:
df["Month"] = pd.Categorical(df["Month"], categories=MONTH_ORDER, ordered=True)

full_grid = pd.MultiIndex.from_product([MONTH_ORDER, SEASON_ORDER], names=["Month", "Season"]).to_frame(index=False)
present = df[["Month", "Season"]].drop_duplicates()
missing = full_grid.merge(present, how="left", indicator=True).query("_merge == 'left_only'")[["Month", "Season"]]

placeholders = missing.assign(Unit_Price=np.nan, Revenue=np.nan, Quantity=3,
                              Product_Name="", State="")
anim_df = pd.concat([df, placeholders], ignore_index=True)
anim_df["Month"] = pd.Categorical(anim_df["Month"], categories=MONTH_ORDER, ordered=True)
anim_df = anim_df.sort_values(["Month", "Season"])

fig5 = px.scatter(anim_df, x="Unit_Price", y="Revenue", color="Season", size="Quantity",
                  animation_frame="Month",
                  category_orders={"Season": SEASON_ORDER, "Month": MONTH_ORDER},
                  color_discrete_map=SEASON_COLORS,
                  hover_name="Product_Name", hover_data={"State": True, "Quantity": True, "Month": False},
                  size_max=22, range_x=[0, df["Unit_Price"].max() * 1.05],
                  range_y=[0, df["Revenue"].max() * 1.05],
                  labels={"Unit_Price": "Unit price (INR)", "Revenue": "Revenue (INR)", "Season": "Season"},
                  title="Unit price vs. revenue by month (press play); marker size = quantity")
fig5.update_layout(legend_itemsizing="constant", height=600, width=980, margin=dict(t=70))
fig5.update_traces(marker=dict(sizemin=3))
fig5.layout.updatemenus[0].buttons[0].args[1]["frame"]["duration"] = 700
fig5.show()

In [ ]:
# Orders per month, used in the comparison below
df.groupby("Month", observed=True).size().reindex(MONTH_ORDER).rename("orders").to_frame().T

Month,January,February,March,April,May,June,July,August,September,October,November,December
orders,120,114,122,110,120,115,113,104,113,180,173,96


### 2.2 One chart, dropdown to switch the y-axis metric

In [ ]:
metrics = {
    "Revenue":         ("Revenue (INR)", "linear"),
    "Delivery_Days":   ("Delivery days", "linear"),
    "Customer_Rating": ("Customer rating (1-5)", "linear"),
    "Discount":        ("Discount (fraction)", "linear"),
}

colors = df["Season"].map(SEASON_COLORS)
fig6 = go.Figure(go.Scatter(
    x=df["Unit_Price"], y=df["Revenue"], mode="markers",
    marker=dict(color=colors, size=8, opacity=0.65, line=dict(width=0.5, color="white")),
    text=df["Product_Name"], customdata=df["Season"],
    hovertemplate="<b>%{text}</b><br>Unit price: %{x}<br>Value: %{y}<br>Season: %{customdata}<extra></extra>",
))

buttons = []
for col, (label, axis_type) in metrics.items():
    buttons.append(dict(
        label=col,
        method="update",
        args=[{"y": [df[col]]},
              {"yaxis.title.text": label, "yaxis.type": axis_type,
               "title.text": f"Unit price vs. {col}"}],
    ))

fig6.update_layout(
    title="Unit price vs. Revenue",
    height=600, width=980, margin=dict(t=110),
    xaxis_title="Unit price (INR)", yaxis_title=metrics["Revenue"][0],
    updatemenus=[dict(buttons=buttons, direction="down", x=1.0, xanchor="right",
                      y=1.15, yanchor="top", showactive=True)],
    annotations=[dict(text="y-axis:", x=0.915, xref="paper", y=1.12, yref="paper",
                      showarrow=False, xanchor="right")],
)
fig6.show()

In [ ]:
# Correlations: what the dropdown lets you compare against Unit_Price
df[["Unit_Price", "Revenue", "Delivery_Days", "Customer_Rating", "Discount"]].corr(method="spearman").loc["Unit_Price"].round(2)

Unit_Price         1.00
Revenue            0.89
Delivery_Days     -0.00
Customer_Rating   -0.04
Discount          -0.00
Name: Unit_Price, dtype: float64

*Switching the dropdown shows unit price is strongly related to `Revenue` (as expected from the revenue formula) but essentially unrelated to `Delivery_Days`, `Customer_Rating` or `Discount` — price does not predict how fast an order ships, how happy the customer is, or how big a discount they received.*

### Comparison: animated (temporal, multivariate) vs. a single static frame (univariate)

*What the animation shows that a single static frame cannot:*

1. A static frame (one month, or one variable such as the Revenue histogram) shows ~120 orders with a stable price–revenue relationship and a cluster of low-price, low-quantity orders. Nothing in a single frame hints at a month-to-month demand shift.

2. Playing the animation adds time as a dimension and reveals:
   - The price–revenue diagonal relationship is stable every month: it is a mathematical property of the revenue formula, not a monthly artefact.
   - Season colours rotate predictably (Winter: Dec–Feb, Summer: Mar–Jun, Monsoon: Jul–Sep, Festive: Oct–Nov), confirming the season mapping is correctly tied to `Order_Date` rather than randomly assigned.
   - October–November (Festive) frames show a visibly denser cluster of larger, higher-value markers, consistent with festive-season shopping behaviour (bigger baskets and higher quantities) that a single-month static view would miss entirely.

**Conclusion:** the univariate view (e.g. the Revenue histogram) describes the overall distribution; the animated multivariate view tests whether that distribution is stable across the year and exposes seasonal demand patterns that must be accounted for in any monthly forecasting or inventory model.